# Spark Environment
This notebook includes examples of how to connect and configure Spark to run in a local environment and to manage the dependencies and options required to read/write data to an S3 comptabile object storage.

### Environment Notes
* By convention credentials and secrets are injected into the runtime as environment variables.
  * BASH environment variables are usually named in all caps.
  * Scala variables are generally lowercase.
* AWS/S3 variables are prefixed using `OBJECTS_*`

## Configure Runtime
Jupyter can be configured to work with [BeakerX](http://beakerx.com/), a set of kernels and extensions which provide JVM support, Spark cluster support, and interactive plots/tables/forms.

* `%%classpath`: One of BeakerX's "magics" provides support for dynamically loading dependencies through the use of Maven.

In [ ]:
%%classpath add mvn
org.apache.hadoop hadoop-common 3.1.2
org.apache.spark spark-sql_2.11 2.4.5
org.apache.spark spark-kubernetes_2.11 2.4.5
tech.tablesaw tablesaw-core 0.37.3
tech.tablesaw tablesaw-beakerx 0.37.3
tech.tablesaw tablesaw-json 0.37.3
tech.tablesaw tablesaw-jsplot 0.37.3
com.amazonaws aws-java-sdk-s3 1.11.744

Packages:

* Spark SQL: Spark's structured data interface
* Spark Kubernetes: Set of tools and utilities that allows Spark to run workers on Kubernetes.
* [TableSaw](https://github.com/jtablesaw/tablesaw): Flexible dataframe and visualization classes for Java and other JRE languages. (Rough equivalent of Pandas in the Python ecosystem.)
* Amazon AWS Java SDK for Amazon S3

In [ ]:
%classpath add jar /opt/spark/jars/joda-time-2.9.9.jar
%classpath add jar /opt/spark/jars/httpclient-4.5.3.jar
%classpath add jar /opt/spark/jars/aws-java-sdk-s3-1.11.534.jar
%classpath add jar /opt/spark/jars/aws-java-sdk-kms-1.11.534.jar
%classpath add jar /opt/spark/jars/aws-java-sdk-dynamodb-1.11.534.jar
%classpath add jar /opt/spark/jars/aws-java-sdk-core-1.11.534.jar
%classpath add jar /opt/spark/jars/aws-java-sdk-1.11.534.jar
%classpath add jar /opt/spark/jars/hadoop-aws-3.1.2.jar
%classpath add jar /opt/spark/jars/slf4j-api-1.7.29.jar
%classpath add jar /opt/spark/jars/slf4j-log4j12-1.7.29.jar

_JARS included in the commands above are needed to allow Spark to read and write data from MinIO. They are included as part of the Spark dependencies in the container._

In [ ]:
%import tech.tablesaw.api.*
%import tech.tablesaw.columns.*
%import tech.tablesaw.beakerx.*

// Display TableSaw tables with BeakerX table display widget
tech.tablesaw.beakerx.TablesawDisplayer.register()

## Working With Data
Getting data loaded is often the hardest part of data analysis. Scala provides flexible classes and methods to help facilitate data loading from sources such as remote files on the Internet, or secure file storage such as S3 object storage.

* Retrieve files from the Internet
* Read/write data to S3 compatible object storage

In [ ]:
// Example 1: Download URL contents to a String in Scala
import scala.io.Source
import java.net.URL
import tech.tablesaw.api.{Table}
import tech.tablesaw.io.csv.{CsvReadOptions}

// Retrieve remote data and convert the results to a string
val surl1 = "https://oak-tree.tech/documents/96/framingham.csv"
val d1 = Source.fromURL(surl1)
val dstr1 = d1.mkString

// View first ten lines of string
dstr1.split("[\\r\\n]+").take(10).foreach(println)

### Working with Object Storage
_Example adapted from the [MinIO cookbook](https://github.com/minio/cookbook/blob/master/docs/aws-sdk-for-java-with-minio.md)._

#### Upload String Data to S3 Object

In [ ]:
// Imports required to work with Buckets
import com.amazonaws.ClientConfiguration
import com.amazonaws.auth.{
    BasicAWSCredentials,AWSStaticCredentialsProvider}
import com.amazonaws.services.s3.AmazonS3Client
import com.amazonaws.client.builder.AwsClientBuilder

import com.amazonaws.regions.Regions
import com.amazonaws.services.s3.{AmazonS3,AmazonS3ClientBuilder}
import com.amazonaws.services.s3.model.Bucket
import com.amazonaws.services.s3.model.{
    GetObjectRequest,PutObjectRequest, S3Object}


// Retrieve credentials from environment
val OBJECT_STORAGE_URL = sys.env.getOrElse("OBJECTS_ENDPOINT", "")
val OBJECT_STORAGE_ACCESSID = sys.env.getOrElse("OBJECTS_ACCESSID", "")
val OBJECT_STORAGE_SECRET = sys.env.getOrElse("OBJECTS_SECRET", "")
val OBJECTS_PLAYGROUND_BUCKET = "playground"
val OBJECTS_PREFIX = sys.env.getOrElse("HOSTNAME", "")


// S3 Access Credentials and Client Configuration
println(s"Object Storage URL: $OBJECT_STORAGE_URL")

// Client Configfuration
val awsconf = new ClientConfiguration()
awsconf.setSignerOverride("AWSS3V4SignerType")

// Credentials
val awsauth = new BasicAWSCredentials(
    OBJECT_STORAGE_ACCESSID, OBJECT_STORAGE_SECRET)

In [ ]:
// Initialize S3 Client Instance
val s3 = AmazonS3ClientBuilder
    .standard()
    .withEndpointConfiguration(
        new AwsClientBuilder.EndpointConfiguration(
            OBJECT_STORAGE_URL, Regions.US_EAST_1.name()))
    .withPathStyleAccessEnabled(true)
    .withClientConfiguration(awsconf)
    .withCredentials(new AWSStaticCredentialsProvider(awsauth))
    .build()

In [ ]:
// The "key" name is created from the playground bucket and 

// container hostname to prevent collissions on shared infrastructure.
var fkey = s"$OBJECTS_PREFIX/framingham.csv"
println(s"File key: $fkey")

// Upload the data from the CSV retrieved earlier to the bucket.
s3.putObject(OBJECTS_PLAYGROUND_BUCKET, fkey, dstr1)

### Working With Structured Data: HTTP(S)
Tablesaw implements a compact data structure called a data frame. Tablesaw implements dataframes around a "structured data" model with columns and rows. See the [tablesaw tutorial](https://jtablesaw.github.io/tablesaw/gettingstarted.html) and [documentation](https://www.javadoc.io/static/tech.tablesaw/tablesaw-core/0.34.0/index.html) for additional detail.

* Columns are named, one-deminsional collections of data.
  - All data in a column must be of the same type.
  - Tablesaw provides support for a variety of primitive structures: Strings, floats, doubles, ints, shorts, longs, bool, and dates.

In [ ]:
// Example 2: Retrieve CSV file and initialize TableSaw DataFrame
val durl = new URL(surl1)
val options1 = CsvReadOptions.builder(durl).header(true)

// Retrieve data and visualize the structure/schema of the dataframe
val dt1 = Table.read().usingOptions(options1)
dt1.structure()

* Dataframes in TableSaw share a great deal in common with dataframes in Spark. It includes methods to help you understand the structure of the data and the values.
  - `structure`: Show a copy of the schema (as interpreted by the dataframe upon load, or provided)
  - `shape`: Provide information about the number of rows (observations) and columns (variables)
  - `first(n)`: Retrieve and display the first `n` rows of the data frame
  - `last(n)`: Retrieve and display the last `n` rows of the data frame

In [ ]:
// View Framingham Data
dt1.first(10)

In [ ]:
// Calculate descriptives and other information 
dt1.summary()

* Descriptive statistics help to provide context about what types of variables the table contains.
  - Some variables are continuously distributed: age, sysBP, diaBP
  - Others are categorical in nature: cigsPerDay, BPMeds, Male

In [ ]:
// Use filters to cross-tabulate information about patients who currently smoke

// TableSaw includes a rich set of methods that allow data to be filtered in
// a functional manner
import tech.tablesaw.api.QuerySupport.{and,or,not}
import tech.tablesaw.aggregate.AggregateFunctions._

// Retrieve all rows where the patient is a smoker
val smokers = dt1.where(dt1.intColumn("currentSmoker").isEqualTo(1))

// Summarize number of cigarettes, systolic blood pressure, disastolic blood pressure
// by the educational level
var smoker_education  = smokers.summarize("cigsPerDay", mean, sum, min, max).by("education")

### Working With Structured Data: S3

In [ ]:
// Use S3 client to retrieve data from object storage
import scala.io.Source

// Example 1: Retrieve object contents and loop over first set of lines
// managing the IO using the Scala standard libraryb
val d2 = s3.getObject(new GetObjectRequest(OBJECTS_PLAYGROUND_BUCKET, fkey))

// Check file contents by retrieving ten lines and iterating over
// their contents. Note use of "take". This is a standard method on 
// many Scala collections and returns the first N items. It's useful
// for truncating a subset of data you wish to inspect.
Source.fromInputStream(d2.getObjectContent)
    .getLines.take(10).foreach(println)

* `take` is an important method on Scala's collection data structures. It takes an integer as the first parameter and returns a new collection consisting of the first N elements.

In [ ]:
// Example 2: Retrieve data from the object storage and create a table saw table
val d2 = s3.getObject(new GetObjectRequest(OBJECTS_PLAYGROUND_BUCKET, fkey))

val options = CsvReadOptions.builder(d2.getObjectContent).header(true)
val dt2 = Table.read().usingOptions(options)

***

## Exercise: Stage DOT Flight Data to MinIO
Prior to being able to analyze data in Spark, if must be staged to a specific location that is accessible to the workers (executors) which will process it. Often this will be to a data lake or storage cluster (such as S3 object storage or HDFS). 

The following files will be utilized throughout the remainder of the course. Write a simple program to download the data and stage it within the MinIO cluster in this enviornment so that Spark is able to consume it from the provided bucket/file keys.

In [ ]:
val FLIGHTS_DATAFILES = Map(
    "web-age/wa2843/airports.json" -> "https://oak-tree.tech/documents/103/airports.json",
    "web-age/wa2843/dot.flights.2018.json" -> "https://oak-tree.tech/documents/105/dot.flights.2018.json",
    "web-age/wa2843/dot.flights.2017-0102.json" -> "https://oak-tree.tech/documents/104/dot.flights.2017-0102.json"
)

// Reminder/Hint: Iteration over map
for ((k,v) <- FLIGHTS_DATAFILES) println(s"File key: $k\nSource URL: $v\n")

***

## Input/Output With Spark
Spark is able to read data from a large number of sources and write data to a large number of sinks.

* Many of the storage classes are provided by Hadoop, showing an example of where Spark builds on top of the development done by other projects.

### Configure Spark Context and Context
_See [Spark documentation](https://spark.apache.org/docs/latest/configuration.html) for a reference of configuration values and options._

In [ ]:
// Import SparkConf and SparkContext
import org.apache.spark.{SparkConf,SparkContext}
import org.apache.spark.sql.SparkSession

// Create configuration 
val conf = new SparkConf()
    .set("spark.driver.memory", "2G")
    .set("spark.executor.memory", "1G")
    .set("spark.driver.maxResultSize", "1G")
    .set("spark.driver.host", sys.env.getOrElse("HOSTNAME", ""))

// Storage configuration
conf.set("spark.hadoop.fs.s3a.endpoint", OBJECT_STORAGE_URL)
    .set("spark.hadoop.fs.s3a.access.key", OBJECT_STORAGE_ACCESSID)
    .set("spark.hadoop.fs.s3a.secret.key", OBJECT_STORAGE_SECRET)
    .set("spark.hadoop.fs.s3a.fast.upload", "true")
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
    .set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

// Set application master and hostname. The local[2] means run two threads "minimal"
// parallelism which allows for bugs to be located.
conf.setMaster("local[2]")
    .setAppName("iotest")

// Create the Spark Context and Session
val sc = new SparkContext(conf)
val spark = SparkSession.builder()
    .appName("iotest").getOrCreate()

In [ ]:
// Import Spark implicits, which allow the use of $-notation
import spark.implicits._

#### Check that Spark Session is Active

In [ ]:
// Create a distributed data to test the session
val t = sc.parallelize(1 to 1000 by 2)

// Pull data from workers to foreground thread
t.take(10).foreach(println)

#### Write Data to MinIO
Spark is able to read and write data to object storage by specifying the file path details in the URL. This has the general form:

`s3a://{{ bucket }}/{{ object-key-path }}`. Example: `s3a://oak-tree.tech/hello-test.txt`

In [ ]:
// Create S3 URL to write test data to
val s3_testurl = s"s3a://$OBJECTS_PLAYGROUND_BUCKET/$OBJECTS_PREFIX/colors-test.csv"
println(s3_testurl)

In [ ]:
// Create data frame from sequence/rdd. Apply column names
val cnames = Seq("name", "color")
val df1 = spark.createDataFrame(
        sc.parallelize(Seq(("Mario", "Red"), ("Luigi", "Green"), ("Princess", "Pink"))))
    .toDF(cnames: _*)

df1.show()

In [ ]:
// Write data to object storage
df1.write.mode("overwrite").option("header", true).csv(s3_testurl)

#### Read Data from MinIO

In [ ]:
// Read data as structured CSV
val df2 = spark.read.format("csv")
    .option("inferSchema", "true")
    .option("header", "true")
    .option("sep", ",")
    .load(s3_testurl)

df2.show()